In [8]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
from epiweeks import Week
from datetime import timedelta
from tqdm import tqdm 
from lmfit import Parameters
import lmfit as lm

In [ ]:
file_path = "../data/raw/dengue.csv.gz"
data_casos = pd.read_csv(file_path)
display(data_casos.head())
print(f"Columns: {data_casos.columns}")

,geocode,date,casos,epiweek,uf,macroregional_geocode,regional_geocode,uf_code,target_city,train_1,target_1,train_2,target_2,train_3,target_3,train_4,target_4,disease
0,3301900,2010-01-03,5,201001,RJ,3312,33006,33,False,True,False,True,False,True,False,True,False,dengue
1,1504505,2010-01-03,0,201001,PA,1512,15014,15,False,True,False,True,False,True,False,True,False,dengue
2,1503077,2010-01-03,0,201001,PA,1511,15008,15,False,True,False,True,False,True,False,True,False,dengue
3,2106409,2010-01-03,0,201001,MA,2110,21006,21,False,True,False,True,False,True,False,True,False,dengue
4,3135704,2010-01-03,0,201001,MG,3103,31024,31,False,True,False,True,False,True,False,True,False,dengue


Columns: Index(['geocode', 'date', 'casos', 'epiweek', 'uf', 'macroregional_geocode',
       'regional_geocode', 'uf_code', 'target_city', 'train_1', 'target_1',
       'train_2', 'target_2', 'train_3', 'target_3', 'train_4', 'target_4',
       'disease'],
      dtype='str')


In [10]:
# see the min and max dates
print(f"Min date: {data_casos['date'].min()}")
print(f"Max date: {data_casos['date'].max()}")

Min date: 2010-01-03
Max date: 2026-03-08


In [11]:
MIN_CASOS_TEMPORADA = 50 
VAL_YEARS = range(2011, 2026) 

results_list = []

In [12]:
grouped = data_casos.groupby('geocode')

In [19]:
# Richards Model
@np.vectorize
def richards(L, a, b, t, tj):
    """Function to apply the Richards models"""
    j = L - L * (1 + a * np.exp(b * (t - tj))) ** (-1 / a)
    return j


def obj_fun(params, aux, df):
    """Objective function"""
    window = df.shape[0]
    pars = params.valuesdict()
    L = pars["L1"]
    tp = pars["tp1"]
    a = pars["a1"]
    b = pars["b1"]

    t_range = np.arange(window)
    richfun = richards(L, a, b, t_range, tp)
    serie = df.casos_cum.values

    mse = (serie - richfun) ** 2 / window

    return mse


def get_SIR_pars(rp: dict):
    """
    Returns the SIR parameters based on the Richards model's parameters (rp)
    """
    a = rp["a1"]
    b = rp["b1"]
    tc = rp["tp1"]
    pars = {
        "beta": b / a,
        "gamma": (b / a) - b,
        "R0": (b / a) / ((b / a) - b),
        "tc": tc,
    }
    return pars


def comp_duration(curve, tj):
    """
    Versão corrigida para evitar erro de tamanho (ValueError)
    """
    # 1. Criamos o df_aux com base no tamanho real da curva passada
    df_aux = pd.DataFrame()
    
    # Garantimos que os tipos sejam datetime
    curve["data_iniSE"] = pd.to_datetime(curve["data_iniSE"])
    
    # Usamos a curva inteira para o df_aux, não apenas os primeiros 52
    df_aux["dates"] = curve["data_iniSE"].values
    df_aux["SE"] = [Week.fromdate(i).cdcformat() for i in df_aux["dates"]]

    # 2. Cálculo da diferença (ajuste de tamanho dinâmico)
    # np.diff retorna n-1 elementos. Concatenamos com um [0] no início.
    diff_values = np.diff(curve.richards.values)
    df_aux["diff_richards"] = np.concatenate(([0], diff_values))

    # 3. Resto da lógica de pico
    pw_date = Week.fromstring(str(df_aux.SE[0])).startdate() + timedelta(
        days=7
    ) * int(round(tj, 0))
    pw = Week.fromdate(pw_date).cdcformat()

    max_c = df_aux["diff_richards"].max()
    # Evita divisão por zero se max_c for 0
    threshold = 0.05 * max_c if max_c > 0 else 0
    
    df_aux_filtered = df_aux.loc[df_aux.diff_richards >= threshold].sort_index()

    if not df_aux_filtered.empty:
        ini = str(df_aux_filtered["SE"].values[0])
        end = str(df_aux_filtered["SE"].values[-1])
        
        ini_aux = Week.fromstring(ini).startdate()
        end_aux = Week.fromstring(end).startdate()
        dur = int((end_aux - ini_aux).days / 7)

        t_ini = np.where(curve.data_iniSE.values == df_aux_filtered.dates.values[0])[0][0]
        t_end = np.where(curve.data_iniSE.values == df_aux_filtered.dates.values[-1])[0][0]
    else:
        ini, end, dur, t_ini, t_end = np.nan, np.nan, 0, np.nan, np.nan

    # Verificação de segurança para o pico
    if not pd.isna(end) and (Week.fromstring(str(pw)).startdate() - Week.fromstring(end).startdate()).days >= 0:
        # Se o pico estimado estiver fora da janela de dados
        pass 

    return {
        "ini": ini,
        "pw": pw,
        "end": end,
        "dur": dur,
        "t_ini": t_ini,
        "t_end": t_end,
    }


def otim(df, verbose=False):
    """Function to optimize the parameters"""
    df.reset_index(inplace=True)
    df["casos_cum"] = df.casos_est.cumsum()
    sum_cases = df.casos_est.sum()
    params = Parameters()
    params.add("gamma", min=0.3, max=0.33)
    params.add("L1", min=1.0, max=1.2 * sum_cases)
    params.add("tp1", min=5, max=35)
    params.add("b1", min=1e-6, max=1)
    params.add("a1", expr="b1/(gamma + b1)", min=0.001, max=1)

    t_range = np.arange(df.shape[0])

    out = lm.minimize(
        obj_fun, params, args=(0, df), method="diferential_evolution"
    )
    if verbose:
        if out.success:
            print(f"found  match after {out.nfev} tries")
        else:
            print("No match found")
            return False, df

    pars = out.params
    pars = pars.valuesdict()

    # serie = df.loc[t_ini:t_fin].casos_cum.values
    richfun_opt = richards(
        pars["L1"], pars["a1"], pars["b1"], t_range, pars["tp1"]
    )

    df = df.copy()
    df.loc[:, "richards"] = richfun_opt + np.zeros(df.shape[0])

    return out, df

In [21]:
for geocode, df_city in tqdm(grouped, desc="Processando municípios"):
    # Garantir que o index é a data para facilitar o slice
    df_city = df_city.set_index(pd.to_datetime(df_city['date'])).sort_index()
    
    for year in VAL_YEARS:
        # Definição da Temporada: EW 41 (Ano anterior) até EW 40 (Ano atual)
        # Conforme a metodologia da competição
        
        start_date = pd.to_datetime(Week(year - 1, 41).startdate())
        end_date = pd.to_datetime(Week(year, 40).startdate())
        
        # Filtra o recorte da temporada
        df_season = df_city.loc[start_date:end_date].copy()
        
        # Verificação de Critério de Transmissão (Scan)
        # Se houver poucos casos, pulamos para evitar ruído no modelo
        if df_season['casos'].sum() < MIN_CASOS_TEMPORADA:
            continue
            
        # Preparação para a função otim (Richards)
        # A função otim espera uma coluna 'casos_est' e 'data_iniSE'
        df_season = df_season.rename(columns={'casos': 'casos_est', 'date': 'data_iniSE'})

        # Dentro do seu loop, antes de chamar a função otim:
        df_season['data_iniSE'] = pd.to_datetime(df_season['data_iniSE']) # Garante o tipo datetime
        
            # Executa a otimização de Richards
        out, curve_df = otim(df_season)
            
        if out.success:
                # Extração dos parâmetros SIR (beta, gamma, R0)
                sir_pars = get_SIR_pars(out.params.valuesdict())
                
                # Cálculo de duração e semanas de pico
                tp1_val = out.params['tp1'].value
                ep_dur = comp_duration(curve_df, tp1_val)
                
                # Cálculo de Erro (RMSE) para controle de qualidade
                rmse = np.sqrt(out.chisqr / len(df_season))
                
                # Salva no dicionário de resultados
                results_list.append({
                    'geocode': geocode,
                    'season_end_year': year,
                    'disease': df_city['disease'].iloc[0],
                    'total_cases_L1': out.params['L1'].value,
                    'peak_week': ep_dur['pw'],
                    'R0': sir_pars['R0'],
                    'beta': sir_pars['beta'],
                    'gamma': sir_pars['gamma'],
                    'duration_weeks': ep_dur['dur'],
                    'rmse': rmse,
                    'success': True
                })
        else:
            print(f"Município {geocode} - Temporada {year}: Otimização falhou, pulando...")
            continue

# Transformando em DataFrame final
df_episcanner_results = pd.DataFrame(results_list)

# Exemplo de visualização
print(df_episcanner_results.head())

Processando municípios:  41%|████▏     | 2309/5570 [23:53<1:05:33,  1.21s/it]

Município 3106200 - Temporada 2024: Otimização falhou, pulando...


Processando municípios:  58%|█████▊    | 3242/5570 [35:14<24:37,  1.58it/s]  

Município 3304557 - Temporada 2011: Otimização falhou, pulando...
Município 3304557 - Temporada 2012: Otimização falhou, pulando...


Processando municípios:  58%|█████▊    | 3243/5570 [35:22<1:53:58,  2.94s/it]

...

Município 3550308 - Temporada 2015: Otimização falhou, pulando...


Processando municípios:  97%|█████████▋| 5417/5570 [48:22<01:53,  1.35it/s]

Município 5208707 - Temporada 2013: Otimização falhou, pulando...
Município 5208707 - Temporada 2022: Otimização falhou, pulando...


Processando municípios: 100%|██████████| 5570/5570 [50:02<00:00,  1.86it/s]

   geocode  season_end_year disease  total_cases_L1 peak_week        R0  \
0  1100015             2012  dengue       37.113773    201224  1.392005   
1  1100015             2013  dengue      165.198386    201305  2.193093   
2  1100015             2016  dengue      566.030215    201604  2.238394   
3  1100015             2019  dengue      125.644570    201924  2.133822   
4  1100015             2020  dengue       68.862082    201947  2.276911   

       beta     gamma  duration_weeks       rmse  success  
0  0.417601  0.300000              50   1.075852     True  
1  0.657928  0.300000              19   0.980334     True  
2  0.738670  0.330000              17   5.228477     True  
3  0.640147  0.300000              20  17.630583     True  
4  0.751046  0.329853              13   0.131021     True  


In [24]:
len(results_list)

22792

In [32]:
df_episcanner_results

,geocode,season_end_year,disease,total_cases_L1,peak_week,R0,beta,gamma,duration_weeks,rmse,success
0,1100015,2012,dengue,37.113773,201224,1.392005,0.417601,0.300000,50,1.075852,True
1,1100015,2013,dengue,165.198386,201305,2.193093,0.657928,0.300000,19,0.980334,True
2,1100015,2016,dengue,566.030215,201604,2.238394,0.738670,0.330000,17,5.228477,True
3,1100015,2019,dengue,125.644570,201924,2.133822,0.640147,0.300000,20,17.630583,True
4,1100015,2020,dengue,68.862082,201947,2.276911,0.751046,0.329853,13,0.131021,True
...,...,...,...,...,...,...,...,...,...,...,...
22787,5300108,2021,dengue,16730.811765,202118,1.377908,0.454709,0.330000,48,1609.196227,True
22788,5300108,2022,dengue,66775.222933,202216,1.624562,0.487368,0.300000,36,21597.325131,True
22789,5300108,2023,dengue,31918.947747,202316,1.352643,0.405793,0.300000,48,12278.051671,True
22790,5300108,2024,dengue,286693.905915,202409,1.906336,0.571901,0.300000,26,875072.782906,True


In [ ]:
chifile_path = "../data/raw/chikungunya.csv.gz"
chidata_casos = pd.read_csv(chifile_path)
display(chidata_casos.head())
print(f"Columns: {chidata_casos.columns}")

,geocode,date,casos,epiweek,uf,macroregional_geocode,regional_geocode,uf_code,target_city,train_1,target_1,train_2,target_2,train_3,target_3,train_4,target_4,disease
0,1200013,2013-12-29,0,201401,AC,1201,12002,12,False,True,False,True,False,True,False,True,False,chikungunya
1,4312955,2013-12-29,0,201401,RS,4311,43020,43,False,True,False,True,False,True,False,True,False,chikungunya
2,4312906,2013-12-29,0,201401,RS,4310,43025,43,False,True,False,True,False,True,False,True,False,chikungunya
3,2924652,2013-12-29,0,201401,BA,2917,29006,29,False,True,False,True,False,True,False,True,False,chikungunya
4,4312807,2013-12-29,0,201401,RS,4310,43025,43,False,True,False,True,False,True,False,True,False,chikungunya


Columns: Index(['geocode', 'date', 'casos', 'epiweek', 'uf', 'macroregional_geocode',
       'regional_geocode', 'uf_code', 'target_city', 'train_1', 'target_1',
       'train_2', 'target_2', 'train_3', 'target_3', 'train_4', 'target_4',
       'disease'],
      dtype='str')


In [34]:
# see the min and max dates
print(f"Min date: {chidata_casos['date'].min()}")
print(f"Max date: {chidata_casos['date'].max()}")

Min date: 2013-12-29
Max date: 2026-03-08


In [35]:
CHI_MIN_CASOS_TEMPORADA = 50 
CHI_VAL_YEARS = range(2015, 2026) 

results_list = []

In [36]:
chi_grouped = chidata_casos.groupby('geocode')

In [37]:
for geocode, df_city in tqdm(chi_grouped, desc="Processando municípios (Chikungunya)"):
    # Garantir que o index é a data para facilitar o slice
    df_city = df_city.set_index(pd.to_datetime(df_city['date'])).sort_index()
    
    for year in CHI_VAL_YEARS:
        # Definição da Temporada: EW 41 (Ano anterior) até EW 40 (Ano atual)
        # Conforme a metodologia da competição
        
        start_date = pd.to_datetime(Week(year - 1, 41).startdate())
        end_date = pd.to_datetime(Week(year, 40).startdate())
        # Filtra o recorte da temporada
        df_season = df_city.loc[start_date:end_date].copy()
        # Verificação de Critério de Transmissão (Scan)
        # Se houver poucos casos, pulamos para evitar ruído no modelo
        if df_season['casos'].sum() < CHI_MIN_CASOS_TEMPORADA:
            continue
        # Preparação para a função otim (Richards)
        # A função otim espera uma coluna 'casos_est' e 'data_iniSE'
        df_season = df_season.rename(columns={'casos': 'casos_est', 'date': 'data_iniSE'})
        df_season['data_iniSE'] = pd.to_datetime(df_season['data_iniSE']) # Garante o tipo datetime
        
            # Executa a otimização de Richards
        out, curve_df = otim(df_season)

        if out.success:
                # Extração dos parâmetros SIR (beta, gamma, R0)
                sir_pars = get_SIR_pars(out.params.valuesdict())
                
                # Cálculo de duração e semanas de pico
                tp1_val = out.params['tp1'].value
                ep_dur = comp_duration(curve_df, tp1_val)
                
                # Cálculo de Erro (RMSE) para controle de qualidade
                rmse = np.sqrt(out.chisqr / len(df_season))
                
                # Salva no dicionário de resultados
                results_list.append({
                    'geocode': geocode,
                    'season_end_year': year,
                    'disease': df_city['disease'].iloc[0],
                    'total_cases_L1': out.params['L1'].value,
                    'peak_week': ep_dur['pw'],
                    'R0': sir_pars['R0'],
                    'beta': sir_pars['beta'],
                    'gamma': sir_pars['gamma'],
                    'duration_weeks': ep_dur['dur'],
                    'rmse': rmse,
                    'success': True
                })
        else:
            print(f"Município {geocode} - Temporada {year}: Otimização falhou, pulando...")
            continue

# Transformando em DataFrame final
df_chi_episcanner_results = pd.DataFrame(results_list)
# Exemplo de visualização
print(df_chi_episcanner_results.head())

Processando municípios (Chikungunya): 100%|██████████| 5570/5570 [05:45<00:00, 16.12it/s] 

   geocode  season_end_year      disease  total_cases_L1 peak_week        R0  \
0  1100015             2025  chikungunya      129.338212    202523  2.013743   
1  1100023             2025  chikungunya      130.068290    202519  1.746855   
2  1100049             2016  chikungunya       76.227663    201607  2.352276   
3  1100049             2025  chikungunya       65.260762    202517  1.409188   
4  1100056             2016  chikungunya       93.047362    201606  2.632797   

       beta     gamma  duration_weeks      rmse  success  
0  0.604343  0.300109              23  0.287902     True  
1  0.576096  0.329791              27  0.388613     True  
2  0.712330  0.302826              17  0.102722     True  
3  0.450111  0.319412              46  0.108905     True  
4  0.789839  0.300000              14  0.118167     True  


In [38]:
len(results_list)

2885

In [39]:
df_chi_episcanner_results

,geocode,season_end_year,disease,total_cases_L1,peak_week,R0,beta,gamma,duration_weeks,rmse,success
0,1100015,2025,chikungunya,129.338212,202523,2.013743,0.604343,0.300109,23,0.287902,True
1,1100023,2025,chikungunya,130.068290,202519,1.746855,0.576096,0.329791,27,0.388613,True
2,1100049,2016,chikungunya,76.227663,201607,2.352276,0.712330,0.302826,17,0.102722,True
3,1100049,2025,chikungunya,65.260762,202517,1.409188,0.450111,0.319412,46,0.108905,True
4,1100056,2016,chikungunya,93.047362,201606,2.632797,0.789839,0.300000,14,0.118167,True
...,...,...,...,...,...,...,...,...,...,...,...
2880,5300108,2021,chikungunya,209.859390,202123,1.436897,0.465985,0.324300,45,1.650880,True
2881,5300108,2022,chikungunya,741.442689,202217,1.511632,0.453489,0.300000,43,6.955693,True
2882,5300108,2023,chikungunya,793.683641,202316,1.463599,0.439080,0.300000,44,6.995103,True
2883,5300108,2024,chikungunya,548.165429,202412,1.461165,0.438349,0.300000,40,3.855464,True


In [ ]:
# agrupar os dataframes de resultados por geocode e temporada, e salvar em um arquivo csv
final_results = pd.concat([df_episcanner_results, df_chi_episcanner_results], ignore_index=True)
final_results.to_csv("../data/processed/episcanner  _results_2026.csv", index=False)